# 01 · Data profiling — Spotify

Requirements this profiling supports (point 6.1):

| ID | Analytical requirement |
|---|---|
| **AR-01** | Grammy penetration by Spotify genre |
| **AR-02** | Era of recognition: popularity and audio profile of Grammy artists by decade |
| **AR-03** | Grammy category family vs artist popularity |

The final table of the notebook lists the quality risks found (`RK-SP-xx`). Every number in it is computed by the code above it.

In [ ]:
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)

SPOTIFY_CSV = "../data/raw/spotify/spotify_dataset.csv"
RISKS_CSV = "../docs/evidence/profiling/spotify_risk_table.csv"   # the final risk table is saved here

risks = []   # the risk table is filled while we profile


def add_risk(condition, risk_id, attribute, evidence, risk, dimension, requirement):
    """Add a row to the risk table only if the profiling really shows the problem."""
    if condition:
        risks.append({
            "Risk ID": risk_id,
            "Dataset / Attribute": attribute,
            "Profiling Evidence": evidence,
            "Potential Quality Risk": risk,
            "Quality Dimension": dimension,
            "Related Requirement": requirement,
        })

## 1 · Structure
What does the file contain? Rows, columns, data types and distinct values.

In [ ]:
df = pd.read_csv(SPOTIFY_CSV)
print("rows x columns:", df.shape)
df.head(3)

In [ ]:
structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "distinct": df.nunique(),
})
structure

In [ ]:
index_cols = [c for c in df.columns if c.startswith("Unnamed")]
data_cols = [c for c in df.columns if c not in index_cols]
print("Redundant index column(s) exported with the file:", index_cols or "none")

add_risk(len(index_cols) > 0, "RK-SP-01", ", ".join(index_cols),
         f"Column(s) {index_cols} are a row index exported with the CSV.",
         "A technical column without analytical meaning that could be mistaken for a business key.",
         "Structure", "— (informational)")

## 2 · Completeness
Are there missing values? A missing artist cannot be linked to Grammy.

In [ ]:
nulls = df.isna().sum()
print(nulls[nulls > 0] if nulls.any() else "No null values")

key_cols = ["artists", "album_name", "track_name"]
rows_with_nulls = df[df[key_cols].isna().any(axis=1)]
print(f"\nrows with a null in {key_cols}: {len(rows_with_nulls)}")
rows_with_nulls.head()

In [ ]:
add_risk(len(rows_with_nulls) > 0, "RK-SP-02", "artists, album_name, track_name",
         f"{len(rows_with_nulls)} row(s) ({len(rows_with_nulls) / len(df):.4%} of the file) with a null in {key_cols}; null artists: {df['artists'].isna().sum()}.",
         "A track without artist cannot be linked to any Grammy artist and would be left out of every artist-level KPI.",
         "Completeness", "AR-01, AR-02, AR-03")

## 3 · Uniqueness and duplicates
Which column identifies a row? What does one row represent? This decides the grain of the fact table.

In [ ]:
n_rows = len(df)
n_tracks = df["track_id"].nunique()
print(f"rows: {n_rows:,} | distinct track_id: {n_tracks:,} | rows that repeat a track_id: {n_rows - n_tracks:,}")

repeated_pairs = df.duplicated(["track_id", "track_genre"]).sum()
exact_duplicates = df.duplicated(data_cols).sum()
print("repeated (track_id, track_genre) pairs:", repeated_pairs)
print("exact duplicate rows (all columns):", exact_duplicates)

genres_per_track = df.groupby("track_id")["track_genre"].nunique()
tracks_in_many_genres = (genres_per_track > 1).sum()
print("tracks listed under more than one genre:", tracks_in_many_genres)

In [ ]:
# When a track_id repeats, do its rows carry the same popularity and audio values?
value_cols = ["popularity", "duration_ms", "danceability", "energy", "loudness", "speechiness",
              "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
repeated_rows = df[df.duplicated("track_id", keep=False)]
differs = repeated_rows.groupby("track_id")[value_cols].nunique() > 1
tracks_with_conflict = differs.any(axis=1).sum()
print("tracks whose repeated rows have different values:", tracks_with_conflict)
print(differs.sum()[differs.sum() > 0] if tracks_with_conflict else "All repeated rows carry identical values.")

In [ ]:
add_risk(n_rows != n_tracks, "RK-SP-03", "track_id, track_genre",
         f"{n_rows:,} rows but only {n_tracks:,} distinct track_id ({n_rows - n_tracks:,} repeated rows); "
         f"{tracks_in_many_genres:,} tracks appear under more than one genre; repeated (track_id, track_genre) pairs: {repeated_pairs:,}.",
         "One row is a track listed under a genre, not a track: track_id is not a primary key and counting rows as tracks inflates every count.",
         "Uniqueness", "AR-01, AR-02, AR-03")
add_risk(tracks_with_conflict > 0, "RK-SP-04", "popularity and audio features of repeated track_id",
         f"{tracks_with_conflict:,} repeated tracks have different values between their rows.",
         "The same track would have more than one value for a track-level measure; a rule to choose one value is needed.",
         "Consistency", "AR-02, AR-03")
add_risk(exact_duplicates > 0, "RK-SP-05", "all columns",
         f"{exact_duplicates:,} exact duplicate rows.",
         "Duplicated rows would repeat facts in the Data Warehouse.",
         "Uniqueness", "AR-01, AR-02, AR-03")

## 4 · The `artists` column
It is the only link with Grammy, so we check its format: several artists in one row? spelling variants?

In [ ]:
def clean_name(text):
    """Lower case, no accents, no punctuation, single spaces. Example: 'Beyoncé ' -> 'beyonce'."""
    if pd.isna(text):
        return None
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^\w\s]", " ", text.lower().replace("&", " and "))
    return re.sub(r"\s+", " ", text).strip() or None


artist_count = df["artists"].dropna().str.split(";").map(len)
multi_artist_rows = (artist_count > 1).sum()
print(f"rows with more than one artist: {multi_artist_rows:,} ({multi_artist_rows / len(artist_count):.1%}) | max artists in one row: {artist_count.max()}")

# one row per credited artist (this is the grain of the Spotify fact table)
credits = (df[["track_id", "track_genre", "artists"]].dropna()
           .assign(artist=lambda d: d["artists"].str.split(";"))
           .explode("artist"))
credits["artist_key"] = credits["artist"].map(clean_name)
print("artist credits after splitting the column:", f"{len(credits):,}")

In [ ]:
spellings = credits.groupby("artist_key")["artist"].nunique()
names_with_variants = (spellings > 1).sum()
print("distinct artist names (raw):", f"{credits['artist'].nunique():,}", "| after cleaning:", f"{credits['artist_key'].nunique():,}")
print("cleaned names written in more than one way:", names_with_variants)
credits[credits["artist_key"].isin(spellings[spellings > 1].index)].groupby("artist_key")["artist"].unique().head(8)

In [ ]:
add_risk(multi_artist_rows > 0, "RK-SP-06", "artists",
         f"{multi_artist_rows:,} of {len(artist_count):,} rows ({multi_artist_rows / len(artist_count):.1%}) have more than one artist separated by ';' "
         f"(up to {artist_count.max()}); splitting gives {len(credits):,} artist credits.",
         "The column has several values: comparing the whole text with Grammy misses collaborations, and a duo is treated as a different entity from its members.",
         "Validity", "AR-01, AR-02, AR-03")
add_risk(names_with_variants > 0, "RK-SP-07", "artists",
         f"{names_with_variants:,} cleaned artist names appear with more than one spelling (case, accents, spaces or punctuation).",
         "Spelling variants of the same artist would become several artists in the dimension and would break the match with Grammy names.",
         "Consistency", "AR-01, AR-02, AR-03")

## 5 · Genres
AR-01 compares genres, so we check how many there are and how balanced they are.

In [ ]:
genre_counts = df["track_genre"].value_counts()
print("distinct genres:", len(genre_counts))
print(genre_counts.describe().round(1)[["min", "50%", "max"]])
genre_counts.plot(kind="bar", figsize=(14, 3), title="Rows per genre")
plt.show()

variation = genre_counts.std() / genre_counts.mean()
print(f"coefficient of variation of rows per genre: {variation:.3f}")

In [ ]:
add_risk(variation < 0.25, "RK-SP-08", "track_genre",
         f"{len(genre_counts)} genres with a similar number of rows each (min {genre_counts.min():,}, median {genre_counts.median():,.0f}, max {genre_counts.max():,}; variation {variation:.3f}; flag: below 0.25).",
         "The file looks sampled per genre, so row or track counts per genre reflect the sampling, not the real size of the genre. AR-01 must use rates (distinct artists), not raw counts.",
         "Representativeness", "AR-01")

## 6 · Numeric columns
Ranges, impossible values and zeros. Audio features and popularity are the measures of AR-02 and AR-03.

In [ ]:
df[value_cols].describe().T.round(3)

In [ ]:
allowed = {                                   # column: (minimum, maximum) expected
    "popularity": (0, 100),
    "danceability": (0, 1), "energy": (0, 1), "speechiness": (0, 1), "acousticness": (0, 1),
    "instrumentalness": (0, 1), "liveness": (0, 1), "valence": (0, 1),
    "loudness": (-60, 0),                     # dB
    "tempo": (1, 250),                        # BPM (0 means "not detected")
    "duration_ms": (1, 1_200_000),            # up to 20 minutes
}
out_of_range = pd.Series({col: int(((df[col] < lo) | (df[col] > hi)).sum()) for col, (lo, hi) in allowed.items()})
print("values outside the expected range:")
print(out_of_range[out_of_range > 0] if out_of_range.any() else "none")

bad_time_signature = (~df["time_signature"].isin([3, 4, 5, 6, 7])).sum()
print("\ntime_signature outside 3-7:", bad_time_signature)

zero_popularity = (df["popularity"] == 0).mean()
print(f"popularity == 0: {zero_popularity:.1%} of the rows")

In [ ]:
df[["popularity", "energy", "danceability", "valence", "acousticness", "loudness"]].hist(bins=40, figsize=(12, 6))
plt.tight_layout()
plt.show()

In [ ]:
problems = out_of_range[out_of_range > 0].to_dict()
add_risk(bool(problems) or bad_time_signature > 0, "RK-SP-09", "tempo, time_signature, duration_ms, loudness, 0-1 features",
         f"values outside the expected range: {problems or 'none'}; time_signature outside 3-7: {bad_time_signature:,}.",
         "Sentinel values (0 = not detected) and extreme durations would bias the audio-profile averages if they were treated as real measurements.",
         "Validity", "AR-02")
add_risk(zero_popularity >= 0.05, "RK-SP-10", "popularity",
         f"{zero_popularity:.1%} of the rows have popularity = 0 (flag: 5% or more).",
         "A large number of zeros mixes 'not popular' with 'no signal' and pulls the averages down; the treatment of zeros must be decided.",
         "Validity", "AR-02, AR-03")

## 7 · Dates
AR-02 needs a timeline. Does the Spotify file have one?

In [ ]:
date_like = [c for c in df.columns if re.search(r"date|year|release|month|created|updated", c, re.I)]
print("date-like columns:", date_like or "none")

add_risk(not date_like, "RK-SP-11", "(whole file)",
         f"None of the {len(df.columns)} columns is a date, a year or a timestamp.",
         "Spotify rows cannot be placed on the Grammy timeline: popularity and audio features describe the artist's catalogue today, not at the time of the award.",
         "Timeliness", "AR-02, AR-03")

## 8 · Size of the dimensional model
These numbers size the dimensions and the Spotify fact table of point 6.4.

In [ ]:
fact_rows = len(credits.drop_duplicates(["track_id", "track_genre", "artist_key"]))
sizing = pd.Series({
    "distinct tracks  (dim_track)": n_tracks,
    "distinct genres  (dim_genre)": df["track_genre"].nunique(),
    "distinct artists after cleaning  (dim_artist, Spotify side)": credits["artist_key"].nunique(),
    "track x genre listings": len(df.drop_duplicates(["track_id", "track_genre"])),
    "rows of fact_track_credit (track x genre x artist)": fact_rows,
}, name="rows")
sizing

## 9 · Risk table (Spotify)
Required by point 6.3: *Dataset / Attribute · Profiling Evidence · Potential Quality Risk · Related Requirement* (plus an ID and the quality dimension, which is used in point 6.5).

In [ ]:
risk_table = pd.DataFrame(risks)
risk_table

In [ ]:
Path(RISKS_CSV).parent.mkdir(parents=True, exist_ok=True)
risk_table.to_csv(RISKS_CSV, index=False)
print(f"{len(risk_table)} risks found. Saved to {RISKS_CSV}")

## How these findings are used next
* **Grain and keys (6.4, 6.8):** `RK-SP-03`, `RK-SP-04`, `RK-SP-05` → one fact row per track × genre × artist, and a rule to remove duplicates.
* **Artist integration key (6.8):** `RK-SP-06`, `RK-SP-07` → split `artists` on `;` and clean the names before matching with Grammy.
* **Quality rules and thresholds (6.5):** `RK-SP-02`, `RK-SP-09`, `RK-SP-10` → completeness and range rules for the raw validation gate.
* **Analysis limits:** `RK-SP-08`, `RK-SP-11` → AR-01 uses rates, and the era analysis (AR-02) can only use the Grammy year.